# Navier--Stokes Re=500 -- U-NO-H10, 7-block, 40-step BPTT, 70-epoch budget

**Default study:** The default executes only N=200: earlier N=25, 50 and 100 jobs already have results. Set TRAIN_SIZES=[25,50,100,200] for a fresh, consistent 70-epoch sweep. This run does not reuse the 500-epoch checkpoints because the training budget/configuration changed.

**Protocol:** 10 observed states; 40-step differentiable autoregressive training; 70 maximum epochs; validation every 5 epochs; best checkpoint selected exclusively by validation forecast error; final test evaluated once per training size. Epochs are not optimizer steps.

**Metrics:** physical absolute space--time L2 over forecast frames 10--100 (excluding observed inputs), final-time L2, first-forecast L2, per-trajectory L2, error-vs-time curve, validation curve, optimizer updates, and training-epoch elapsed time. Saved under the unique OUTPUT_ROOT on Google Drive.

**Important:** H10 has ten ground-truth input frames and is NOT information-matched with single-state AWFNO. Do not compare forecast-only H10 errors to H1 full-horizon errors without an explicitly matched evaluation.

**Execution status:** This notebook contains code only; no GPU runs have been performed or results invented.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, math, os, random, time, gc, csv
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
import matplotlib.pyplot as plt

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda':print('WARNING: 40-step training requires substantial GPU compute.')
torch.backends.cuda.matmul.allow_tf32=False
random.seed(1234);np.random.seed(1234);torch.manual_seed(1234)
if torch.cuda.is_available():torch.cuda.manual_seed_all(1234)


In [ ]:

CONFIG={
 'DATA_PATH':'/content/drive/MyDrive/Kolmogorov_NS_Re500_start0.25_horizon0.25_dt0.0025_v1/kolmogorov_dataset.pt',
 'OUTPUT_ROOT':'/content/drive/MyDrive/AWFNO_NS_Re500_101_UNO_H10_7BLOCK_40STEP_70E_JCP',
 'TRAIN_SIZES':[200], 'TRAIN_POOL':200, 'N_VAL':30, 'N_TEST':70,
 'SPLIT_MODE':'contiguous', 'SEED':1234, 'SPLIT_SEED':2026,
 'HISTORY':10, 'TRAIN_STEPS':40, 'STRIDE':40,
 'WIDTH':32,'FACTOR':0.75,'MODES':[11,7,3,3,3,7,11],'NORMALIZE_BLOCKS':False,
 'EPOCHS':70,'BATCH_SIZE':1,'LR':5e-4,'WEIGHT_DECAY':1e-5,
 'LOSS':'relative_l2', # original Cahn--Hilliard UNO objective; choose normalized_mse for an extra study
 'VAL_EVERY':5,'PATIENCE_EVALS':12,'SAVE_EVERY':5,
 'GRAD_CLIP':1.0, 'ACTIVATION_CHECKPOINTING':True,
 'EVAL_BATCH_SIZE':2,'SEED_EACH_N':True, 'FORCE_RETRAIN':False,
 'TEST_ONLY_AFTER_TRAINING':True,
}

ROOT=Path(CONFIG['OUTPUT_ROOT']);ROOT.mkdir(parents=True,exist_ok=True)
assert CONFIG['HISTORY']==10 and CONFIG['TRAIN_STEPS']==40
assert max(CONFIG['TRAIN_SIZES'])<=CONFIG['TRAIN_POOL']
H=CONFIG['HISTORY'];STEPS=CONFIG['TRAIN_STEPS']

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def calculate_train_stats(X,n):
    # Streaming float64 statistics from the N training trajectories ONLY.
    total=total2=count=0
    for i in range(n):
        a=np.asarray(X[i],dtype=np.float64)
        total+=a.sum(); total2+=np.square(a).sum();count+=a.size
    mean=total/count;std=np.sqrt(max(total2/count-mean**2,1e-12))
    return float(mean),float(std)

class HistoryWindowDataset(Dataset):
    def __init__(self,trajectories,n,history=10,horizon=40,stride=40):
        self.U=trajectories;self.n=n;self.h=history;self.k=horizon
        self.starts=list(range(0,self.U.shape[-1]-history-horizon+1,stride))
        final=self.U.shape[-1]-history-horizon
        if final not in self.starts:self.starts.append(final)
        assert self.starts and min(self.starts)>=0
    def __len__(self):return self.n*len(self.starts)
    def __getitem__(self,i):
        uid=i//len(self.starts);s=self.starts[i%len(self.starts)]
        # H10 inputs (t=s,...,s+9), 40 future targets (t=s+10,...,s+49).
        x=torch.as_tensor(np.array(self.U[uid,...,s:s+self.h],copy=True),dtype=torch.float32)
        y=torch.as_tensor(np.array(self.U[uid,...,s+self.h:s+self.h+self.k],copy=True),dtype=torch.float32)
        return x,y

def real_dof(model):
    return sum(p.numel()*(2 if p.is_complex() else 1) for p in model.parameters() if p.requires_grad)

# Differentiable 40-step rollout loss, selectable relative L2 or normalized MSE.
# Relative L2 matches the Cahn--Hilliard H10 training objective.
def train_batch(model,history,truth,mean,std,optimizer):
    model.train();history=(history.to(DEVICE)-mean)/std
    targets=(truth.to(DEVICE)-mean)/std
    optimizer.zero_grad(set_to_none=True)
    total=torch.zeros((),device=DEVICE)
    for j in range(targets.shape[-1]):
        if CONFIG['ACTIVATION_CHECKPOINTING']:
            pred=checkpoint(model,history,use_reentrant=False)
        else:pred=model(history)
        if CONFIG['LOSS']=='relative_l2':
            pred_phys=pred*std+mean;truth_phys=targets[...,j]*std+mean
            pp=(pred_phys-truth_phys).reshape(pred.shape[0],-1)
            tt=truth_phys.reshape(pred.shape[0],-1)
            step_loss=(pp.norm(dim=-1)/(tt.norm(dim=-1)+1e-12)).mean()
        elif CONFIG['LOSS']=='normalized_mse':
            step_loss=F.mse_loss(pred,targets[...,j])
        else:raise ValueError(CONFIG['LOSS'])
        total=total+step_loss/targets.shape[-1]
        history=torch.cat((history[...,1:],pred.unsqueeze(-1)),dim=-1)
    if not torch.isfinite(total):raise FloatingPointError('nonfinite training loss')
    total.backward()
    grad=torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['GRAD_CLIP'])
    if not torch.isfinite(grad):raise FloatingPointError('nonfinite gradient')
    optimizer.step()
    return float(total.detach())

@torch.inference_mode()
def predict(model, U, mean,std,batch=2):
    model.eval();shape=U.shape;results=np.empty(shape,dtype=np.float32)
    results[...,:H]=U[...,:H] # only the ten OBSERVED frames
    for start in range(0,len(U),batch):
        v=torch.as_tensor(np.asarray(U[start:start+batch,...,:H]).copy(),device=DEVICE,dtype=torch.float32)
        v=(v-mean)/std
        for t in range(H,U.shape[-1]):
            z=model(v)
            if not torch.isfinite(z).all():raise FloatingPointError(f'nonfinite prediction t={t}')
            results[start:start+len(v),...,t]=(z*std+mean).cpu().numpy()
            v=torch.cat([v[...,1:],z.unsqueeze(-1)],dim=-1)
    return results

def save_state_atomic(obj,path):
    tmp=path.with_suffix('.tmp');torch.save(obj,tmp);tmp.replace(path)

def training_loop(model,train,validation,mean,std,n,geometry):
    out=ROOT/f'N{n}_seed{CONFIG["SEED"]}';out.mkdir(exist_ok=True,parents=True)
    config_path=out/'configuration.json';checkpoint_path=out/'last_checkpoint.pt';best_path=out/'best.pt'
    cfg=dict(CONFIG);cfg['n_train']=n;cfg['geometry']=geometry
    cfg['data_size_bytes']=Path(CONFIG['DATA_PATH']).stat().st_size
    cfg_id=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:14]
    if config_path.exists():
        assert json.loads(config_path.read_text())['config_hash']==cfg_id, 'Checkpoint/config mismatch; use a distinct OUTPUT_ROOT'
    config_path.write_text(json.dumps({'config_hash':cfg_id,**cfg},indent=2))
    seed_all(CONFIG['SEED']+n if CONFIG['SEED_EACH_N'] else CONFIG['SEED'])
    # Initialization happens BEFORE this call. For exact seed control the caller
    # seeds immediately BEFORE constructing model.
    opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['LR'],weight_decay=CONFIG['WEIGHT_DECAY'])
    ds=HistoryWindowDataset(train,n,H,STEPS,CONFIG['STRIDE'])
    loader=DataLoader(ds,batch_size=CONFIG['BATCH_SIZE'],shuffle=True,num_workers=0,pin_memory=True)
    first=1;best=np.inf;best_epoch=0;stale=0;logs=[]
    if checkpoint_path.exists() and not CONFIG['FORCE_RETRAIN']:
        state=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
        assert state['config_hash']==cfg_id
        model.load_state_dict(state['model']);opt.load_state_dict(state['opt'])
        first=state['epoch']+1;best=state['best'];best_epoch=state['best_epoch'];stale=state['stale'];logs=state['logs']
        if 'rng_cpu' in state:torch.set_rng_state(state['rng_cpu'])
        if DEVICE.type=='cuda' and state.get('rng_cuda') is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        print('RESUME:',out,'epoch',first)
    for epoch in range(first,CONFIG['EPOCHS']+1):
        t0=time.perf_counter();losses=[]
        for h,f in loader:losses.append(train_batch(model,h,f,mean,std,opt))
        row={'epoch':epoch,'train_rollout_loss':float(np.mean(losses)),'seconds':time.perf_counter()-t0}
        if epoch==1 or epoch%CONFIG['VAL_EVERY']==0 or epoch==CONFIG['EPOCHS']:
            predicted=predict(model,validation,mean,std,CONFIG['EVAL_BATCH_SIZE'])
            scores=metric_report(predicted,validation,report_prefix='val')
            row.update({k:v for k,v in scores.items() if isinstance(v,(int,float))})
            # NO testing or test-based selection: validation forecast t>=10.
            current=scores['forecast_l2_mean']
            if current<best:
                best=current;best_epoch=epoch;stale=0
                save_state_atomic({'model':model.state_dict(),'cfg':cfg,'epoch':epoch,'best_validation_forecast_l2':best},best_path)
            else:stale+=1
            print(f'N={n} epoch={epoch} train={row["train_rollout_loss"]:.6g} val_forecast={current:.6g} best={best:.6g}@{best_epoch}',flush=True)
        logs.append(row)
        if epoch%CONFIG['SAVE_EVERY']==0 or epoch==CONFIG['EPOCHS'] or stale>=CONFIG['PATIENCE_EVALS']:
            save_state_atomic({'config_hash':cfg_id,'model':model.state_dict(),'opt':opt.state_dict(),'epoch':epoch,
                               'best':best,'best_epoch':best_epoch,'stale':stale,'logs':logs,
                               'rng_cpu':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None},checkpoint_path)
        if stale>=CONFIG['PATIENCE_EVALS']:
            print('EARLY STOP',epoch,'best',best_epoch);break
    assert best_path.exists(), 'Missing best validation checkpoint'
    model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=False)['model'])
    (out/'training_log.json').write_text(json.dumps(logs,indent=2))
    return out,best,best_epoch,logs

def run_experiments():
    summary=[]
    for n in CONFIG['TRAIN_SIZES']:
        seed_all(CONFIG['SEED']+n if CONFIG['SEED_EACH_N'] else CONFIG['SEED'])
        model=UNO2DHistory(X_COORDS,Y_COORDS,GEOMETRY,CONFIG).to(DEVICE)
        params=sum(p.numel() for p in model.parameters() if p.requires_grad)
        print('START',GEOMETRY,'N=',n,'params=',params,'real_dof=',real_dof(model),flush=True)
        mean,std=calculate_train_stats(TRAIN_POOL,n)
        out,best,best_epoch,logs=training_loop(model,TRAIN_POOL,VAL,mean,std,n,GEOMETRY)
        # Run held-out test exactly once per finished N, not at every checkpoint.
        t0=time.perf_counter();pred=predict(model,TEST,mean,std,CONFIG['EVAL_BATCH_SIZE']);seconds=time.perf_counter()-t0
        scores=metric_report(pred,TEST,report_prefix='test')
        row={'model':'UNO2D-H10-7block-40step','PDE':GEOMETRY,'n_train':n,'params':params,
             'real_dof':real_dof(model),'best_epoch':best_epoch,'val_forecast_l2':best,
             'test_seconds':seconds,'forecast_start_index':H,'observed_frames':H,'train_loss':'40_step_'+CONFIG['LOSS'],**scores}
        # All reported values use the held-out 70 trajectories; H10 frames 0..9 are observed.
        # Save physical, absolute L2 versus time and per-trajectory metrics for JCP plots.
        import pandas as pd
        diagnostics=out/'diagnostics';diagnostics.mkdir(parents=True,exist_ok=True)
        l2_time=np.empty((len(TEST),len(TIMES)-H),dtype=np.float64)
        for case in range(len(TEST)):
            diff=np.asarray(pred[case,...,H:]-TEST[case,...,H:],dtype=np.float64)
            l2_time[case]=np.sqrt(np.einsum('ijt,ij->t',diff*diff,SPATIAL_WEIGHTS,optimize=True))
        time_weights=np.ones(l2_time.shape[1],dtype=np.float64)
        time_weights[0]=time_weights[-1]=0.5
        absolute_forecast=np.sqrt(np.sum(l2_time**2*time_weights[None,:],axis=1)*DT)
        pd.DataFrame({'case_id':np.arange(len(TEST)),'forecast_l2':absolute_forecast,
                      'final_l2':l2_time[:,-1],'first_forecast_l2':l2_time[:,0]}
                    ).to_csv(diagnostics/'per_trajectory_metrics.csv',index=False)
        pd.DataFrame({'frame':np.arange(H,len(TIMES)),'time':TIMES[H:],
                      'l2_mean':l2_time.mean(0),'l2_sd':l2_time.std(0)}
                    ).to_csv(diagnostics/'l2_vs_time.csv',index=False)
        fig,ax=plt.subplots(figsize=(7,4))
        ax.plot(TIMES[H:],l2_time.mean(0),lw=1.7)
        ax.fill_between(TIMES[H:],l2_time.mean(0)-l2_time.std(0),
                        l2_time.mean(0)+l2_time.std(0),alpha=.2)
        ax.set(xlabel='Physical time',ylabel='Absolute spatial L2 error',
               title=f'{GEOMETRY}: U-NO-H10 forecast (N={n})')
        fig.tight_layout();fig.savefig(diagnostics/'l2_vs_time.png',dpi=160);plt.close(fig)
        valid=[r for r in logs if 'val_forecast_l2' in r]
        if valid:
            fig,ax=plt.subplots(figsize=(7,4))
            ax.plot([r['epoch'] for r in valid],[r['val_forecast_l2'] for r in valid],marker='o',ms=3)
            ax.set(xlabel='Epoch',ylabel='Validation forecast L2',title=f'U-NO-H10 validation N={n}')
            fig.tight_layout();fig.savefig(diagnostics/'validation_curve.png',dpi=160);plt.close(fig)
        windows_per_traj=len(HistoryWindowDataset(TRAIN_POOL,1,H,STEPS,CONFIG['STRIDE']))
        completed=logs[-1]['epoch']
        row.update({'epochs_completed':int(completed),
                    'optimizer_steps':int(math.ceil(n*windows_per_traj/CONFIG['BATCH_SIZE'])*completed),
                    'train_epoch_seconds_sum':float(sum(x.get('seconds',0.0) for x in logs)),
                    'n_val':len(VAL),'n_test':len(TEST),
                    'train_windows_per_trajectory':windows_per_traj,
                    'reported_metric':'physical_absolute_L2_forecast_only_frames_10_to_100',
                    'checkpoint_selection':'minimum_validation_forecast_L2',
                    'training_budget_epochs':CONFIG['EPOCHS']})
        path=out/'test_metrics.json';path.write_text(json.dumps(row,indent=2,allow_nan=False))
        summary.append(row);print('TEST',n,row,flush=True)
        del model,pred
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    import pandas as pd
    frame=pd.DataFrame(summary);frame.to_csv(ROOT/'summary.csv',index=False)
    print(frame[['n_train','forecast_l2_mean','final_l2_mean','first_forecast_l2_mean','params']].to_string(index=False))
    return frame


## Load original verified dataset and preserve original split

In [ ]:

path=Path(CONFIG['DATA_PATH']);meta_path=path.parent/'metadata.json'
assert meta_path.is_file(),'Expected original verified Kolmogorov metadata.json'
meta=json.loads(meta_path.read_text())
EXPECTED_DATA_SHA='7030bc43e6331cdf472182ae0ed4f2684c18da8d336ca9ed92c37baf0bfc8a06'
assert meta['status']=='PASS' and meta['sha256']==EXPECTED_DATA_SHA,meta
assert meta['shape']==[300,128,128,101]
raw=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
U=raw['U']
assert isinstance(U,torch.Tensor) and U.shape==(300,128,128,101)
assert torch.equal(raw['train_indices'],torch.arange(200))
assert torch.equal(raw['val_indices'],torch.arange(200,230))
assert torch.equal(raw['test_indices'],torch.arange(230,300))
U=U.numpy()  # mmap-backed when supported: do NOT make a 2-GiB duplicate
X_COORDS=np.asarray(raw['x'],dtype=np.float32).reshape(-1)
Y_COORDS=np.asarray(raw['y'],dtype=np.float32).reshape(-1)
TIMES=np.asarray(raw['times'],dtype=np.float64).reshape(-1)
assert len(TIMES)==101 and np.allclose(np.diff(TIMES),0.0025,atol=2e-7)
DT=float(np.mean(np.diff(TIMES)))
TRAIN_POOL=U[:200];VAL=U[200:230];TEST=U[230:300]
GEOMETRY='periodic'
DX=2*np.pi/128
assert np.allclose(X_COORDS,np.arange(128)*DX,atol=1e-6)
SPATIAL_WEIGHTS=np.full((128,128),DX*DX,dtype=np.float64)
print('Navier--Stokes',U.shape,'train',TRAIN_POOL.shape,'validation',VAL.shape,'test',TEST.shape,'dt',DT)


## Physical metrics (no observed-history contamination)

In [ ]:

def metric_report(pred,true,report_prefix):
    assert pred.shape==true.shape
    # strict forecast-only interval [t_10,...,t_100] (no free observed t0..t9).
    # Chunk to avoid large temporary float64 arrays on 128x128 x 101 grids.
    e2=np.empty((len(pred),true.shape[-1]-H),dtype=np.float64)
    for i in range(0,len(pred),2):
        err=np.asarray(pred[i:i+2,...,H:]-true[i:i+2,...,H:],dtype=np.float64)
        e2[i:i+2]=np.einsum('nijt,ij->nt',err*err,SPATIAL_WEIGHTS,optimize=True)
    wt=np.ones(e2.shape[1],dtype=np.float64);wt[0]=wt[-1]=.5
    full=np.sqrt(np.sum(e2*wt[None,:],axis=1)*DT)
    final=np.sqrt(e2[:,-1]);first=np.sqrt(e2[:,0])
    scores={'forecast_l2_mean':float(np.mean(full)),'forecast_l2_sd':float(np.std(full)),
            'final_l2_mean':float(np.mean(final)),'final_l2_sd':float(np.std(final)),
            'first_forecast_l2_mean':float(np.mean(first)),
            'forecast_interval_start':int(H),'forecast_interval_end':int(true.shape[-1]-1),
            'n_test':int(len(true))}
    return scores


## Exact seven-block spectral U-NO topology (generalized grid/geometry)

In [ ]:
class SpectralConv2dUNO(nn.Module):
    def __init__(
        self, in_channels, out_channels, dim1, dim2, modes1, modes2
    ):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)
        self.modes1 = int(modes1)
        self.modes2 = int(modes2)
        scale = (1.0 / (2.0 * self.in_channels)) ** 0.5
        shape = (
            self.in_channels,
            self.out_channels,
            self.modes1,
            self.modes2,
        )
        self.weights1 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )

    @staticmethod
    def complex_multiply(x, weights):
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        max_modes1 = min(x.shape[-2] // 2, output_dim1 // 2)
        max_modes2 = min(x.shape[-1] // 2 + 1, output_dim2 // 2 + 1)
        if self.modes1 > max_modes1 or self.modes2 > max_modes2:
            raise ValueError(
                f"Modes {(self.modes1, self.modes2)} exceed the admissible "
                f"limit {(max_modes1, max_modes2)} for "
                f"{tuple(x.shape[-2:])} -> {(output_dim1, output_dim2)}."
            )

        # FFT operations remain in float32/complex64; AMP is disabled.
        x_ft = torch.fft.rfft2(x.float(), norm="forward")
        out_ft = torch.zeros(
            x.shape[0],
            self.out_channels,
            output_dim1,
            output_dim2 // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )
        out_ft[:, :, : self.modes1, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, : self.modes1, : self.modes2],
                self.weights1,
            )
        )
        out_ft[:, :, -self.modes1 :, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, -self.modes1 :, : self.modes2],
                self.weights2,
            )
        )
        return torch.fft.irfft2(
            out_ft,
            s=(output_dim1, output_dim2),
            norm="forward",
        )


class PointwiseOp2D(nn.Module):
    def __init__(self, in_channels, out_channels, dim1, dim2):
        super().__init__()
        self.conv = nn.Conv2d(int(in_channels), int(out_channels), 1)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        x = self.conv(x)
        return F.interpolate(
            x,
            size=(output_dim1, output_dim2),
            mode="bicubic",
            align_corners=True,
            antialias=True,
        )


class OperatorBlock2D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        dim1,
        dim2,
        modes1,
        modes2,
        normalize=False,
        nonlinearity=True,
    ):
        super().__init__()
        self.conv = SpectralConv2dUNO(
            in_channels, out_channels, dim1, dim2, modes1, modes2
        )
        self.w = PointwiseOp2D(
            in_channels, out_channels, dim1, dim2
        )
        self.normalize = bool(normalize)
        self.nonlinearity = bool(nonlinearity)
        if self.normalize:
            self.normalizer = nn.InstanceNorm2d(
                int(out_channels), affine=True
            )

    def forward(self, x, dim1=None, dim2=None):
        x = self.conv(x, dim1, dim2) + self.w(x, dim1, dim2)
        if self.normalize:
            x = self.normalizer(x)
        if self.nonlinearity:
            x = F.gelu(x)
        return x




class UNO2DHistory(nn.Module):
    """Seven-block Cahn--Hilliard-style UNO H10 adapted to the target geometry."""
    def __init__(self, x_coords, y_coords, geometry, config):
        super().__init__()
        self.history_length=int(config['HISTORY'])
        self.spatial_shape=(len(x_coords),len(y_coords))
        self.geometry=geometry
        width=int(config['WIDTH']); factor=float(config['FACTOR'])
        modes=list(config['MODES']); normalize=bool(config['NORMALIZE_BLOCKS'])
        nx,ny=self.spatial_shape
        if geometry=='periodic':
            # Periodic sine/cosine channels WITHOUT a duplicated endpoint.
            xx=2*np.pi*np.arange(nx)/nx
            yy=2*np.pi*np.arange(ny)/ny
            X,Y=np.meshgrid(xx,yy,indexing='ij')
            coords=np.stack([np.sin(X),np.cos(X),np.sin(Y),np.cos(Y)],axis=-1)
        elif geometry=='annulus':
            rr=(np.asarray(x_coords,dtype=np.float64)-float(x_coords[0]))/max(float(x_coords[-1]-x_coords[0]),1e-12)
            theta=np.asarray(y_coords,dtype=np.float64)
            R,T=np.meshgrid(rr,theta,indexing='ij')
            coords=np.stack([R, R**2, np.sin(T), np.cos(T)],axis=-1)
        else:raise ValueError(geometry)
        self.register_buffer('coordinates',torch.tensor(coords,dtype=torch.float32)[None],persistent=False)
        self.fc=nn.Linear(self.history_length+4,width//2)
        self.fc0=nn.Linear(width//2,width)
        c0=width;c1=int(2*factor*width);c2=int(4*factor*width);c3=int(8*factor*width)
        # Same relative pyramid as the prior CH H10 model (32 -> 24 -> 16 -> 8).
        # For rectangular domains, each axis is scaled independently.
        pyramid=[(3*nx//4,3*ny//4),(nx//2,ny//2),(nx//4,ny//4)]
        self.pyramid=pyramid
        if min(*pyramid[-1])<8:raise ValueError('Grid too small for seven-block UNO')
        self.L0=OperatorBlock2D(c0,c1,*pyramid[0],modes[0],modes[0],normalize)
        self.L1=OperatorBlock2D(c1,c2,*pyramid[1],modes[1],modes[1],normalize)
        self.L2=OperatorBlock2D(c2,c3,*pyramid[2],modes[2],modes[2],normalize)
        self.L3=OperatorBlock2D(c3,c3,*pyramid[2],modes[3],modes[3],normalize)
        self.L4=OperatorBlock2D(c3,c2,*pyramid[1],modes[4],modes[4],normalize)
        self.L5=OperatorBlock2D(2*c2,c1,*pyramid[0],modes[5],modes[5],normalize)
        self.L6=OperatorBlock2D(2*c1,width,nx,ny,modes[6],modes[6],normalize)
        self.fc1=nn.Linear(2*width,4*width)
        self.fc2=nn.Linear(4*width,1)
    def forward(self,history):
        if history.ndim!=4 or tuple(history.shape[1:3])!=self.spatial_shape or history.shape[-1]!=self.history_length:
            raise ValueError(f'UNO expects (B,{self.spatial_shape[0]},{self.spatial_shape[1]},{self.history_length}), got {tuple(history.shape)}')
        z=torch.cat([history,self.coordinates.expand(history.shape[0],-1,-1,-1)],dim=-1)
        z=F.gelu(self.fc(z)); z0=F.gelu(self.fc0(z)).permute(0,3,1,2)
        a=self.L0(z0);b=self.L1(a);c=self.L2(b);d=self.L3(c)
        e=self.L4(d);e=torch.cat([e,b],dim=1)
        f=self.L5(e);f=torch.cat([f,a],dim=1)
        g=self.L6(f);g=torch.cat([g,z0],dim=1)
        return self.fc2(F.gelu(self.fc1(g.permute(0,2,3,1).contiguous()))).squeeze(-1)


## Smoke tests: input layout, forward output, and differentiable recursion

In [ ]:

# Geometry, differentiability and no-teacher-forcing smoke tests.
seed_all(CONFIG['SEED'])
model=UNO2DHistory(X_COORDS,Y_COORDS,GEOMETRY,CONFIG).to(DEVICE)
assert model.spatial_shape==tuple(TRAIN_POOL.shape[1:3])
sample=torch.as_tensor(np.asarray(TRAIN_POOL[0:1,...,:H]).copy(),device=DEVICE)
with torch.no_grad():z=model(sample)
assert z.shape==tuple(sample.shape[:3]) and torch.isfinite(z).all()
# 2-step gradient smoke test: catches silent FFT/autograd/device errors early.
model.train();sample.requires_grad_(True)
y1=model(sample);hist2=torch.cat([sample[...,1:],y1[...,None]],dim=-1)
y2=model(hist2)
(y1.square().mean()+y2.square().mean()).backward()
assert all(p.grad is not None for p in model.parameters() if p.requires_grad)
print('SMOKE PASS: 7-block H10, 2-step differentiable rollout; model parameters:',sum(p.numel() for p in model.parameters()))
del sample,z,y1,hist2,y2,model
if DEVICE.type=='cuda':torch.cuda.empty_cache()


## 40-step differentiable training, full validation rollouts, best-checkpoint selection

In [ ]:
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def calculate_train_stats(X,n):
    # Streaming float64 statistics from the N training trajectories ONLY.
    total=total2=count=0
    for i in range(n):
        a=np.asarray(X[i],dtype=np.float64)
        total+=a.sum(); total2+=np.square(a).sum();count+=a.size
    mean=total/count;std=np.sqrt(max(total2/count-mean**2,1e-12))
    return float(mean),float(std)

class HistoryWindowDataset(Dataset):
    def __init__(self,trajectories,n,history=10,horizon=40,stride=40):
        self.U=trajectories;self.n=n;self.h=history;self.k=horizon
        self.starts=list(range(0,self.U.shape[-1]-history-horizon+1,stride))
        final=self.U.shape[-1]-history-horizon
        if final not in self.starts:self.starts.append(final)
        assert self.starts and min(self.starts)>=0
    def __len__(self):return self.n*len(self.starts)
    def __getitem__(self,i):
        uid=i//len(self.starts);s=self.starts[i%len(self.starts)]
        # H10 inputs (t=s,...,s+9), 40 future targets (t=s+10,...,s+49).
        x=torch.as_tensor(np.array(self.U[uid,...,s:s+self.h],copy=True),dtype=torch.float32)
        y=torch.as_tensor(np.array(self.U[uid,...,s+self.h:s+self.h+self.k],copy=True),dtype=torch.float32)
        return x,y

def real_dof(model):
    return sum(p.numel()*(2 if p.is_complex() else 1) for p in model.parameters() if p.requires_grad)

# Differentiable 40-step rollout loss, selectable relative L2 or normalized MSE.
# Relative L2 matches the Cahn--Hilliard H10 training objective.
def train_batch(model,history,truth,mean,std,optimizer):
    model.train();history=(history.to(DEVICE)-mean)/std
    targets=(truth.to(DEVICE)-mean)/std
    optimizer.zero_grad(set_to_none=True)
    total=torch.zeros((),device=DEVICE)
    for j in range(targets.shape[-1]):
        if CONFIG['ACTIVATION_CHECKPOINTING']:
            pred=checkpoint(model,history,use_reentrant=False)
        else:pred=model(history)
        if CONFIG['LOSS']=='relative_l2':
            pred_phys=pred*std+mean;truth_phys=targets[...,j]*std+mean
            pp=(pred_phys-truth_phys).reshape(pred.shape[0],-1)
            tt=truth_phys.reshape(pred.shape[0],-1)
            step_loss=(pp.norm(dim=-1)/(tt.norm(dim=-1)+1e-12)).mean()
        elif CONFIG['LOSS']=='normalized_mse':
            step_loss=F.mse_loss(pred,targets[...,j])
        else:raise ValueError(CONFIG['LOSS'])
        total=total+step_loss/targets.shape[-1]
        history=torch.cat((history[...,1:],pred.unsqueeze(-1)),dim=-1)
    if not torch.isfinite(total):raise FloatingPointError('nonfinite training loss')
    total.backward()
    grad=torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['GRAD_CLIP'])
    if not torch.isfinite(grad):raise FloatingPointError('nonfinite gradient')
    optimizer.step()
    return float(total.detach())

@torch.inference_mode()
def predict(model, U, mean,std,batch=2):
    model.eval();shape=U.shape;results=np.empty(shape,dtype=np.float32)
    results[...,:H]=U[...,:H] # only the ten OBSERVED frames
    for start in range(0,len(U),batch):
        v=torch.as_tensor(np.asarray(U[start:start+batch,...,:H]).copy(),device=DEVICE,dtype=torch.float32)
        v=(v-mean)/std
        for t in range(H,U.shape[-1]):
            z=model(v)
            if not torch.isfinite(z).all():raise FloatingPointError(f'nonfinite prediction t={t}')
            results[start:start+len(v),...,t]=(z*std+mean).cpu().numpy()
            v=torch.cat([v[...,1:],z.unsqueeze(-1)],dim=-1)
    return results

def save_state_atomic(obj,path):
    tmp=path.with_suffix('.tmp');torch.save(obj,tmp);tmp.replace(path)

def training_loop(model,train,validation,mean,std,n,geometry):
    out=ROOT/f'N{n}_seed{CONFIG["SEED"]}';out.mkdir(exist_ok=True,parents=True)
    config_path=out/'configuration.json';checkpoint_path=out/'last_checkpoint.pt';best_path=out/'best.pt'
    cfg=dict(CONFIG);cfg['n_train']=n;cfg['geometry']=geometry
    cfg['data_size_bytes']=Path(CONFIG['DATA_PATH']).stat().st_size
    cfg_id=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:14]
    if config_path.exists():
        assert json.loads(config_path.read_text())['config_hash']==cfg_id, 'Checkpoint/config mismatch; use a distinct OUTPUT_ROOT'
    config_path.write_text(json.dumps({'config_hash':cfg_id,**cfg},indent=2))
    seed_all(CONFIG['SEED']+n if CONFIG['SEED_EACH_N'] else CONFIG['SEED'])
    # Initialization happens BEFORE this call. For exact seed control the caller
    # seeds immediately BEFORE constructing model.
    opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['LR'],weight_decay=CONFIG['WEIGHT_DECAY'])
    ds=HistoryWindowDataset(train,n,H,STEPS,CONFIG['STRIDE'])
    loader=DataLoader(ds,batch_size=CONFIG['BATCH_SIZE'],shuffle=True,num_workers=0,pin_memory=True)
    first=1;best=np.inf;best_epoch=0;stale=0;logs=[]
    if checkpoint_path.exists() and not CONFIG['FORCE_RETRAIN']:
        state=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
        assert state['config_hash']==cfg_id
        model.load_state_dict(state['model']);opt.load_state_dict(state['opt'])
        first=state['epoch']+1;best=state['best'];best_epoch=state['best_epoch'];stale=state['stale'];logs=state['logs']
        if 'rng_cpu' in state:torch.set_rng_state(state['rng_cpu'])
        if DEVICE.type=='cuda' and state.get('rng_cuda') is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        print('RESUME:',out,'epoch',first)
    for epoch in range(first,CONFIG['EPOCHS']+1):
        t0=time.perf_counter();losses=[]
        for h,f in loader:losses.append(train_batch(model,h,f,mean,std,opt))
        row={'epoch':epoch,'train_rollout_loss':float(np.mean(losses)),'seconds':time.perf_counter()-t0}
        if epoch==1 or epoch%CONFIG['VAL_EVERY']==0 or epoch==CONFIG['EPOCHS']:
            predicted=predict(model,validation,mean,std,CONFIG['EVAL_BATCH_SIZE'])
            scores=metric_report(predicted,validation,report_prefix='val')
            row.update({k:v for k,v in scores.items() if isinstance(v,(int,float))})
            # NO testing or test-based selection: validation forecast t>=10.
            current=scores['forecast_l2_mean']
            if current<best:
                best=current;best_epoch=epoch;stale=0
                save_state_atomic({'model':model.state_dict(),'cfg':cfg,'epoch':epoch,'best_validation_forecast_l2':best},best_path)
            else:stale+=1
            print(f'N={n} epoch={epoch} train={row["train_rollout_loss"]:.6g} val_forecast={current:.6g} best={best:.6g}@{best_epoch}',flush=True)
        logs.append(row)
        if epoch%CONFIG['SAVE_EVERY']==0 or epoch==CONFIG['EPOCHS'] or stale>=CONFIG['PATIENCE_EVALS']:
            save_state_atomic({'config_hash':cfg_id,'model':model.state_dict(),'opt':opt.state_dict(),'epoch':epoch,
                               'best':best,'best_epoch':best_epoch,'stale':stale,'logs':logs,
                               'rng_cpu':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None},checkpoint_path)
        if stale>=CONFIG['PATIENCE_EVALS']:
            print('EARLY STOP',epoch,'best',best_epoch);break
    assert best_path.exists(), 'Missing best validation checkpoint'
    model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=False)['model'])
    (out/'training_log.json').write_text(json.dumps(logs,indent=2))
    return out,best,best_epoch,logs

def run_experiments():
    summary=[]
    for n in CONFIG['TRAIN_SIZES']:
        seed_all(CONFIG['SEED']+n if CONFIG['SEED_EACH_N'] else CONFIG['SEED'])
        model=UNO2DHistory(X_COORDS,Y_COORDS,GEOMETRY,CONFIG).to(DEVICE)
        params=sum(p.numel() for p in model.parameters() if p.requires_grad)
        print('START',GEOMETRY,'N=',n,'params=',params,'real_dof=',real_dof(model),flush=True)
        mean,std=calculate_train_stats(TRAIN_POOL,n)
        out,best,best_epoch,logs=training_loop(model,TRAIN_POOL,VAL,mean,std,n,GEOMETRY)
        # Run held-out test exactly once per finished N, not at every checkpoint.
        t0=time.perf_counter();pred=predict(model,TEST,mean,std,CONFIG['EVAL_BATCH_SIZE']);seconds=time.perf_counter()-t0
        scores=metric_report(pred,TEST,report_prefix='test')
        row={'model':'UNO2D-H10-7block-40step','PDE':GEOMETRY,'n_train':n,'params':params,
             'real_dof':real_dof(model),'best_epoch':best_epoch,'val_forecast_l2':best,
             'test_seconds':seconds,'forecast_start_index':H,'observed_frames':H,'train_loss':'40_step_'+CONFIG['LOSS'],**scores}
        # All reported values use the held-out 70 trajectories; H10 frames 0..9 are observed.
        # Save physical, absolute L2 versus time and per-trajectory metrics for JCP plots.
        import pandas as pd
        diagnostics=out/'diagnostics';diagnostics.mkdir(parents=True,exist_ok=True)
        l2_time=np.empty((len(TEST),len(TIMES)-H),dtype=np.float64)
        for case in range(len(TEST)):
            diff=np.asarray(pred[case,...,H:]-TEST[case,...,H:],dtype=np.float64)
            l2_time[case]=np.sqrt(np.einsum('ijt,ij->t',diff*diff,SPATIAL_WEIGHTS,optimize=True))
        time_weights=np.ones(l2_time.shape[1],dtype=np.float64)
        time_weights[0]=time_weights[-1]=0.5
        absolute_forecast=np.sqrt(np.sum(l2_time**2*time_weights[None,:],axis=1)*DT)
        pd.DataFrame({'case_id':np.arange(len(TEST)),'forecast_l2':absolute_forecast,
                      'final_l2':l2_time[:,-1],'first_forecast_l2':l2_time[:,0]}
                    ).to_csv(diagnostics/'per_trajectory_metrics.csv',index=False)
        pd.DataFrame({'frame':np.arange(H,len(TIMES)),'time':TIMES[H:],
                      'l2_mean':l2_time.mean(0),'l2_sd':l2_time.std(0)}
                    ).to_csv(diagnostics/'l2_vs_time.csv',index=False)
        fig,ax=plt.subplots(figsize=(7,4))
        ax.plot(TIMES[H:],l2_time.mean(0),lw=1.7)
        ax.fill_between(TIMES[H:],l2_time.mean(0)-l2_time.std(0),
                        l2_time.mean(0)+l2_time.std(0),alpha=.2)
        ax.set(xlabel='Physical time',ylabel='Absolute spatial L2 error',
               title=f'{GEOMETRY}: U-NO-H10 forecast (N={n})')
        fig.tight_layout();fig.savefig(diagnostics/'l2_vs_time.png',dpi=160);plt.close(fig)
        valid=[r for r in logs if 'val_forecast_l2' in r]
        if valid:
            fig,ax=plt.subplots(figsize=(7,4))
            ax.plot([r['epoch'] for r in valid],[r['val_forecast_l2'] for r in valid],marker='o',ms=3)
            ax.set(xlabel='Epoch',ylabel='Validation forecast L2',title=f'U-NO-H10 validation N={n}')
            fig.tight_layout();fig.savefig(diagnostics/'validation_curve.png',dpi=160);plt.close(fig)
        windows_per_traj=len(HistoryWindowDataset(TRAIN_POOL,1,H,STEPS,CONFIG['STRIDE']))
        completed=logs[-1]['epoch']
        row.update({'epochs_completed':int(completed),
                    'optimizer_steps':int(math.ceil(n*windows_per_traj/CONFIG['BATCH_SIZE'])*completed),
                    'train_epoch_seconds_sum':float(sum(x.get('seconds',0.0) for x in logs)),
                    'n_val':len(VAL),'n_test':len(TEST),
                    'train_windows_per_trajectory':windows_per_traj,
                    'reported_metric':'physical_absolute_L2_forecast_only_frames_10_to_100',
                    'checkpoint_selection':'minimum_validation_forecast_L2',
                    'training_budget_epochs':CONFIG['EPOCHS']})
        path=out/'test_metrics.json';path.write_text(json.dumps(row,indent=2,allow_nan=False))
        summary.append(row);print('TEST',n,row,flush=True)
        del model,pred
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
    import pandas as pd
    frame=pd.DataFrame(summary);frame.to_csv(ROOT/'summary.csv',index=False)
    print(frame[['n_train','forecast_l2_mean','final_l2_mean','first_forecast_l2_mean','params']].to_string(index=False))
    return frame


# Exact run configuration; no parameter updates until run_experiments().
print('PDE:',GEOMETRY, '| training sizes:',CONFIG['TRAIN_SIZES'], '| steps:',CONFIG['TRAIN_STEPS'], '| epochs:',CONFIG['EPOCHS'])
print('Checkpoint directory:',ROOT)

In [ ]:
results=run_experiments()
results

**No GPU execution is included.** Run in Colab with Drive mounted. Checkpoints and numerical outputs are produced only after the run completes.